# Homework 2

This notebook contains the code and answer(s) for Problem 2. of Homework 2.

---

Problem 2.) In class we talked about the 3rd and X model.  Using data from the most recent 5 NFL seasons, fit the model described in class and in a well written paragraph evaluate the claim that ``all teams in the NFL are equally good at making 3rd and X.''  Explain any choices that you made regarding the data and the models.

In [23]:
# Import the neccesary Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import nfl_data_py as nfl
import statsmodels.api as sm

In [24]:
# Import the data
nfl_data = nfl.import_pbp_data(years=[2021, 2022, 2023, 2024, 2025])

2021 done.
2022 done.
2023 done.
2024 done.
2025 done.
Downcasting floats.


---

## Notes from class:

**Data:** pbp data, 3rd down data, filter out garbage time, filter out 3rd downs with penalties that move the offense back, filter out end of half data, filter out long 3rd down data (x >= sum #), filter out time remaining in half < 30 secs, filter out own rz, filter out close games (fg takes lead on ‘change in wp’), filter out playoff games, (maybe filter out week 18 - good idea but maybe leave out. This is a ‘choices’ thing).

**(The above are all ideas for filter)**

**Target:** Completed or not

From ‘completed’, it could mean 1st score, or a score, penalties by the defense, 4th down conversion

**predictors/ features:** x = yards to go, offensive/pos team

(above are the HAVE TO HAVE NO MATTER WHAT predictors)

**Form:** linear offense + s(x) + …

s(x) = smooth function

**Modeling choice:** logistic regression (other classification models possible - maybe classification trees but probably not SVM’s -)

**Other predictors:** Defense, home field or not

Likely only do this on data from one year only due to player and coaching changes on certain teams.

Using the notes from above, 2 models will be built. 

- A model with only the 'MUST HAVE' predictors (yards to go (the big X) and offensive pos team)
- And a model including the 'MUST HAVE' predictors as well as the other predictors (Defensive pos team and home field advantage for an offense or not).

---

## Data Filtering

Before modeling the data, I will need to filter the plays to make sure that all of the metrics/ flags we want to use are not null and that they occur on 3rd down.



In [25]:
# Examine the available columns of the play by play data
col_list = nfl_data.columns.tolist()
pd.set_option('display.max_seq_items', None)
print("\n".join(nfl_data.columns))

play_id
game_id
old_game_id_x
home_team
away_team
season_type
week
posteam
posteam_type
defteam
side_of_field
yardline_100
game_date
quarter_seconds_remaining
half_seconds_remaining
game_seconds_remaining
game_half
quarter_end
drive
sp
qtr
down
goal_to_go
time
yrdln
ydstogo
ydsnet
desc
play_type
yards_gained
shotgun
no_huddle
qb_dropback
qb_kneel
qb_spike
qb_scramble
pass_length
pass_location
air_yards
yards_after_catch
run_location
run_gap
field_goal_result
kick_distance
extra_point_result
two_point_conv_result
home_timeouts_remaining
away_timeouts_remaining
timeout
timeout_team
td_team
td_player_name
td_player_id
posteam_timeouts_remaining
defteam_timeouts_remaining
total_home_score
total_away_score
posteam_score
defteam_score
score_differential
posteam_score_post
defteam_score_post
score_differential_post
no_score_prob
opp_fg_prob
opp_safety_prob
opp_td_prob
fg_prob
safety_prob
td_prob
extra_point_prob
two_point_conversion_prob
ep
epa
total_home_epa
total_away_epa
total_home_rush_ep

In [26]:
# Filtering process
nfl_data = nfl_data[
    (nfl_data['down'] == 3) &
    (nfl_data['ydstogo'].notna()) &
    (nfl_data['posteam'].notna()) &
    (nfl_data['defteam'].notna()) &
    (nfl_data['posteam_type'].notna())
]

In [27]:
# Need to make posteam_type into 1s and 0s.
nfl_data['posteam_type'] = nfl_data['posteam_type'].map({'home': 1, 'away': 0})

We will also need to make dummy for both the posteam (offensive team) and the defteam (defensive team)

In [28]:
# Get dummies for posteam and defteam
nfl_data = pd.get_dummies(data=nfl_data, columns=['posteam', 'defteam'], dtype=int)

In [29]:
# Look at the results
# Examine the available columns of the play by play data
nfl_data.columns[395:]

Index(['posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS', 'defteam_ARI', 'defteam_ATL', 'defteam_BAL',
       'defteam_BUF', 'defteam_CAR', 'defteam_CHI', 'defteam_CIN',
       'defteam_CLE', 'defteam_DAL', 'defteam_DEN', 'defteam_DET',
       'defteam_GB', 'defteam_HOU', 'defteam_IND', 'defteam_JAX', 'defteam_KC',
       'defteam_LA', 'defteam_LAC', 'defteam_LV', 'defteam_MIA', 'defteam_MIN',
       'defteam_NE', 'defteam_NO', 'defteam_NYG', 'defteam_NYJ', 'defteam_PHI',
       'defteam_PIT', 'defteam_SEA', 'defteam_SF', 'd

---

### Model with ONLY 'yards to go' and 'offensive pos team':

In [30]:
# Get the predictors
X = nfl_data[['ydstogo', 'posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS']]

# Get the target predictor
y = nfl_data[['posteam_type']]

# add a constant to the data
X = sm.add_constant(X)

# Make the model object
model_1 = sm.Logit(y, X).fit()

# Get the summary
model_1.summary()

Optimization terminated successfully.
         Current function value: 0.692523
         Iterations 4


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           posteam_type   No. Observations:                40112
Model:                          Logit   Df Residuals:                    40079
Method:                           MLE   Df Model:                           32
Date:                Sat, 29 Aug 2026   Pseudo R-squ.:               0.0008978
Time:                        16:17:51   Log-Likelihood:                -27778.
converged:                       True   LL-Null:                       -27803.
Covariance Type:            nonrobust   LLR p-value:                   0.02267
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.0312    6.6e+05   4.74e-08      1.000   -1.29e+06    1.29e+06
ydstogo        -0.0043      0.002     -2.149      0.032      -0.008      -0.000
posteam_ARI    -0.0564    6.6e+05  -8.55e-08      1.000   -1.29e+06    1.29e+06
posteam_ATL    -0.0087    6.6e+05  -1.32e-08      1.000   -1.29e+06    1.29e+06
posteam_BAL     0.0383    6.6e+05   5.81e-08      1.000   -1.29e+06    1.29e+06
posteam_BUF     0.0490    6.6e+05   7.43e-08      1.000   -1.29e+06    1.29e+06
posteam_CAR    -0.0418    6.6e+05  -6.34e-08      1.000   -1.29e+06    1.29e+06
posteam_CHI  3.903e-05    6.6e+05   5.92e-11      1.000   -1.29e+06    1.29e+06
posteam_CIN    -0.0595    6.6e+05  -9.01e-08      1.000   -1.29e+06    1.29e+06
posteam_CLE    -0.0050    6.6e+05  -7.63e-09      1.000   -1.29e+06    1.29e+06
posteam_DAL    -0.0207    6.6e+05  -3.13e-08      1.000   -1.29e+06    1.29e+06
posteam_DEN     0.0694    6.6e+05   1.05e-07      1.000   -1.29e+06    1.29e+06
posteam_DET    -0.0488    6.6e+05  -7.39e-08      1.000   -1.29e+06    1.29e+06
posteam_GB     -0.0790    6.6e+05   -1.2e-07      1.000   -1.29e+06    1.29e+06
posteam_HOU    -0.1050    6.6e+05  -1.59e-07      1.000   -1.29e+06    1.29e+06
posteam_IND    -0.0447    6.6e+05  -6.77e-08      1.000   -1.29e+06    1.29e+06
posteam_JAX     0.0473    6.6e+05   7.16e-08      1.000   -1.29e+06    1.29e+06
posteam_KC      0.1977    6.6e+05      3e-07      1.000   -1.29e+06    1.29e+06
posteam_LA     -0.0154    6.6e+05  -2.34e-08      1.000   -1.29e+06    1.29e+06
posteam_LAC     0.0007    6.6e+05   1.04e-09      1.000   -1.29e+06    1.29e+06
posteam_LV      0.0111    6.6e+05   1.69e-08      1.000   -1.29e+06    1.29e+06
posteam_MIA    -0.0365    6.6e+05  -5.54e-08      1.000   -1.29e+06    1.29e+06
posteam_MIN    -0.0277    6.6e+05  -4.19e-08      1.000   -1.29e+06    1.29e+06
posteam_NE     -0.0190    6.6e+05  -2.88e-08      1.000   -1.29e+06    1.29e+06
posteam_NO     -0.0411    6.6e+05  -6.23e-08      1.000   -1.29e+06    1.29e+06
posteam_NYG     0.0065    6.6e+05   9.77e-09      1.000   -1.29e+06    1.29e+06
posteam_NYJ     0.0730    6.6e+05   1.11e-07      1.000   -1.29e+06    1.29e+06
posteam_PHI     0.1637    6.6e+05   2.48e-07      1.000   -1.29e+06    1.29e+06
posteam_PIT     0.0091    6.6e+05   1.38e-08      1.000   -1.29e+06    1.29e+06
posteam_SEA    -0.0508    6.6e+05   -7.7e-08      1.000   -1.29e+06    1.29e+06
posteam_SF     -0.0344    6.6e+05  -5.21e-08      1.000   -1.29e+06    1.29e+06
posteam_TB      0.0825    6.6e+05   1.25e-07      1.000   -1.29e+06    1.29e+06
posteam_TEN     0.0591    6.6e+05   8.95e-08      1.000   -1.29e+06    1.29e+06
posteam_WAS    -0.0818    6.6e+05  -1.24e-07      1.000   -1.29e+06    1.29e+06
===============================================================================
"""

---

## Model with extra predictors 

In [31]:
# Get the predictors
X = nfl_data[['ydstogo', 'posteam_ARI', 'posteam_ATL', 'posteam_BAL', 'posteam_BUF',
       'posteam_CAR', 'posteam_CHI', 'posteam_CIN', 'posteam_CLE',
       'posteam_DAL', 'posteam_DEN', 'posteam_DET', 'posteam_GB',
       'posteam_HOU', 'posteam_IND', 'posteam_JAX', 'posteam_KC', 'posteam_LA',
       'posteam_LAC', 'posteam_LV', 'posteam_MIA', 'posteam_MIN', 'posteam_NE',
       'posteam_NO', 'posteam_NYG', 'posteam_NYJ', 'posteam_PHI',
       'posteam_PIT', 'posteam_SEA', 'posteam_SF', 'posteam_TB', 'posteam_TEN',
       'posteam_WAS',
       'posteam_type',
       'defteam_ARI', 'defteam_ATL', 'defteam_BAL',
       'defteam_BUF', 'defteam_CAR', 'defteam_CHI', 'defteam_CIN',
       'defteam_CLE', 'defteam_DAL', 'defteam_DEN', 'defteam_DET',
       'defteam_GB', 'defteam_HOU', 'defteam_IND', 'defteam_JAX', 'defteam_KC',
       'defteam_LA', 'defteam_LAC', 'defteam_LV', 'defteam_MIA', 'defteam_MIN',
       'defteam_NE', 'defteam_NO', 'defteam_NYG', 'defteam_NYJ', 'defteam_PHI',
       'defteam_PIT', 'defteam_SEA', 'defteam_SF', 'defteam_TB', 'defteam_TEN',
       'defteam_WAS']]

# Get the target predictor
y = nfl_data[['posteam_type']]

# add a constant to the data
X = sm.add_constant(X)

# Make the model object
model_2 = sm.Logit(y, X).fit()

# Get the summary
model_2.summary()

C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\base\optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\base\optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\base\optimizer.py:536: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  callback(newparams)
C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-pac

         Current function value: 0.000000
         Iterations: 35


C:\Users\ajhay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\discrete\discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           posteam_type   No. Observations:                40112
Model:                          Logit   Df Residuals:                    40047
Method:                           MLE   Df Model:                           64
Date:                Sat, 29 Aug 2026   Pseudo R-squ.:                   1.000
Time:                        16:17:52   Log-Likelihood:            -1.4599e-06
converged:                      False   LL-Null:                       -27803.
Covariance Type:            nonrobust   LLR p-value:                     0.000
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const          -22.1246        nan        nan        nan         nan         nan
ydstogo         -0.0554    165.942     -0.000      1.000    -325.297     325.186
posteam_ARI     -0.6974        nan        nan        nan         nan         nan
posteam_ATL     -0.6902        nan        nan        nan         nan         nan
posteam_BAL     -0.6834        nan        nan        nan         nan         nan
posteam_BUF     -0.6910        nan        nan        nan         nan         nan
posteam_CAR     -0.7012        nan        nan        nan         nan         nan
posteam_CHI     -0.6901        nan        nan        nan         nan         nan
posteam_CIN     -0.7075        nan        nan        nan         nan         nan
posteam_CLE     -0.6842        nan        nan        nan         nan         nan
posteam_DAL     -0.7104        nan        nan        nan         nan         nan
posteam_DEN     -0.6689        nan        nan        nan         nan         nan
posteam_DET     -0.7039        nan        nan        nan         nan         nan
posteam_GB      -0.7191        nan        nan        nan         nan         nan
posteam_HOU     -0.7111        nan        nan        nan         nan         nan
posteam_IND     -0.7006        nan        nan        nan         nan         nan
posteam_JAX     -0.6830        nan        nan        nan         nan         nan
posteam_KC      -0.6569        nan        nan        nan         nan         nan
posteam_LA      -0.7076        nan        nan        nan         nan         nan
posteam_LAC     -0.6977        nan        nan        nan         nan         nan
posteam_LV      -0.6799        nan        nan        nan         nan         nan
posteam_MIA     -0.6872        nan        nan        nan         nan         nan
posteam_MIN     -0.6882        nan        nan        nan         nan         nan
posteam_NE      -0.6956        nan        nan        nan         nan         nan
posteam_NO      -0.6987        nan        nan        nan         nan         nan
posteam_NYG     -0.6861        nan        nan        nan         nan         nan
posteam_NYJ     -0.6586        nan        nan        nan         nan         nan
posteam_PHI     -0.6647        nan        nan        nan         nan         nan
posteam_PIT     -0.6952        nan        nan        nan         nan         nan
posteam_SEA     -0.6924        nan        nan        nan         nan         nan
posteam_SF      -0.7048        nan        nan        nan         nan         nan
posteam_TB      -0.6849        nan        nan        nan         nan         nan
posteam_TEN     -0.6726        nan        nan        nan         nan         nan
posteam_WAS     -0.7117        nan        nan        nan         nan         nan
posteam_type    48.1780   1744.447      0.028      0.978   -3370.875    3467.231
defteam_ARI     -0.6777        nan        nan        nan         nan         nan
defteam_ATL     -0.6830        nan        nan        nan         nan         nan
d